# 04 Index Calculation

Purpose: explain standardization, UEOI weights, yearly rankings, and sensitivity checks.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
scores = pd.read_csv(ROOT / 'data' / 'processed' / 'ueoi_scores.csv')
panel = pd.read_csv(ROOT / 'data' / 'raw' / 'city_panel.csv')
print(f'Scores shape: {scores.shape}')
print(f'UEOI complete: {scores["ueoi_score"].notna().sum()}/{len(scores)}')

## 1. UEOI Formula

$$\text{UEOI} = 0.35 \times \text{Income} + 0.25 \times \text{GDP} + 0.15 \times \text{PopGrowth} + 0.15 \times \text{Innovation} - 0.10 \times \text{HousingBurden}$$

- All components are **min-max standardized to 0–100 within each year**
- `housing_burden_score` is **inverted** (higher = less burden = better)
- Missing components → UEOI set to NaN (no imputation)

## 2. Yearly Rankings

Top 10 cities per year.

In [ ]:
for year in sorted(scores['year'].unique()):
    year_data = scores[scores['year'] == year].dropna(subset=['rank']).sort_values('rank')
    print(f'\n=== {year} Top 10 ===')
    display(year_data[['rank', 'city', 'ueoi_score', 'income_score', 'gdp_score', 
                        'innovation_score', 'housing_burden_score']].head(10))
    if year == scores['year'].max():
        print(f'\n   (4 cities missing rd_expenditure: no UEOI)')

## 3. Score Breakdown: Top vs Bottom Cities

Radar-style visualization of sub-scores.

In [ ]:
latest = scores[(scores['year'] == scores['year'].max()) & scores['ueoi_score'].notna()]
latest = latest.sort_values('ueoi_score', ascending=False)
top5 = latest.head(5)
bottom5 = latest.tail(5)

components = ['income_score', 'gdp_score', 'population_growth_score', 
              'innovation_score', 'housing_burden_score']
comp_labels = ['Income', 'GDP', 'Pop Growth', 'Innovation', 'Housing\n(lower=better)']

fig, axes = plt.subplots(1, 2, figsize=(14, 12), subplot_kw=dict(polar=True))

for ax, title, subset in [(axes[0], 'Top 5 Cities', top5), (axes[1], 'Bottom 5 Cities', bottom5)]:
    angles = np.linspace(0, 2 * np.pi, len(components), endpoint=False).tolist()
    angles += angles[:1]
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(comp_labels, fontsize=9)
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(['20', '40', '60', '80', '100'], fontsize=7)

    for _, row in subset.iterrows():
        values = [row[c] for c in components]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=row['city'], markersize=4)
        ax.fill(angles, values, alpha=0.1)
    ax.set_title(title, fontsize=13, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

plt.suptitle(f'UEOI Sub-Score Profiles ({latest["year"].iloc[0]:.0f})', fontsize=15, y=1.05)
plt.tight_layout()
plt.show()

## 4. Weight Sensitivity Analysis

What if we change the income weight?

In [ ]:
BASE_WEIGHTS = {
    'income_score': 0.35, 'gdp_score': 0.25, 'population_growth_score': 0.15,
    'innovation_score': 0.15, 'housing_burden_score': 0.10
}

def compute_ueoi(df, weights):
    s = sum(df[c] * w for c, w in weights.items())
    return s

latest_valid = scores[scores['ueoi_score'].notna()].copy()

# Test income weights from 0.25 to 0.45
income_weights = np.arange(0.25, 0.50, 0.05)
rank_changes = []

for yr, group in latest_valid.groupby('year'):
    base_ranks = group.sort_values('ueoi_score', ascending=False)['city'].tolist()
    for iw in income_weights:
        alt_weights = dict(BASE_WEIGHTS)
        alt_weights['income_score'] = iw
        alt_weights['gdp_score'] = 1.0 - sum(v for k, v in alt_weights.items() if k != 'gdp_score' and k != 'income_score') - iw
        
        group_alt = group.copy()
        group_alt['alt_ueoi'] = compute_ueoi(group_alt, alt_weights)
        alt_ranks = group_alt.sort_values('alt_ueoi', ascending=False)['city'].tolist()
        
        # Spearman rank correlation
        base_order = {c: i for i, c in enumerate(base_ranks)}
        alt_order = {c: i for i, c in enumerate(alt_ranks)}
        changes = sum(1 for c in base_order if abs(base_order[c] - alt_order.get(c, -1)) > 2)
        rank_changes.append({'year': yr, 'income_weight': iw, 'major_changes': changes})

rc_df = pd.DataFrame(rank_changes)
pivot = rc_df.pivot_table(index='income_weight', columns='year', values='major_changes', aggfunc='first')

fig, ax = plt.subplots(figsize=(8, 5))
pivot.plot(marker='o', ax=ax)
ax.axvline(x=0.35, color='gray', linestyle='--', alpha=0.5, label='Current (0.35)')
ax.set_xlabel('Income Weight')
ax.set_ylabel('Cities with Rank Change > 2 positions')
ax.set_title('Weight Sensitivity: Income Score')
ax.legend(title='Year')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Sensitivity analysis: changing income weight from 0.25→0.45')
print(f'Max rank disruption: {pivot.max().max()} cities with >2 position change')

## 5. Housing Burden Inversion Check

Verify that `housing_burden_score` is correctly inverted: higher = better.

In [ ]:
# Check: city with highest housing_burden should have lowest housing_burden_score
for yr in sorted(panel['year'].unique())[-3:]:
    yr_panel = panel[panel['year'] == yr].dropna(subset=['housing_burden'])
    yr_scores = scores[scores['year'] == yr].dropna(subset=['housing_burden_score'])
    merged = yr_panel.merge(yr_scores[['city', 'year', 'housing_burden_score']], on=['city', 'year'])
    max_burden_city = merged.loc[merged['housing_burden'].idxmax()]
    min_burden_city = merged.loc[merged['housing_burden'].idxmin()]
    print(f'{yr}: Highest burden = {max_burden_city["city"]} ({max_burden_city["housing_burden"]:.4f}, score={max_burden_city["housing_burden_score"]:.1f})')
    print(f'       Lowest burden  = {min_burden_city["city"]} ({min_burden_city["housing_burden"]:.4f}, score={min_burden_city["housing_burden_score"]:.1f})')
    if max_burden_city['housing_burden_score'] < min_burden_city['housing_burden_score']:
        print('       ✅ Inversion correct: higher burden → lower score')
    else:
        print('       ❌ Inversion FAILED')

## 6. UEOI Score Components Weights

Visualize the weight contribution for the top-ranked city.

In [ ]:
top_city = latest.head(1).iloc[0]
contributions = {c: top_city[c] * w for c, w in BASE_WEIGHTS.items()}

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(list(contributions.keys()), list(contributions.values()), 
              color=['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00'], edgecolor='white')
ax.axhline(y=top_city['ueoi_score'], color='darkred', linestyle='--', alpha=0.5, 
           label=f'Total UEOI = {top_city["ueoi_score"]:.1f}')
ax.set_title(f'UEOI Component Contributions: {top_city["city"]} ({top_city["year"]:.0f})')
ax.set_ylabel('Weighted Score')
ax.legend()

for bar, (comp, val) in zip(bars, contributions.items()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f'{val:.1f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()